# Configure Projector
Test out the configuration logic for a projector.

In [1]:
# load some embeddings that we are going to use to configure the projector

from hproj.data.paths import Paths

paths = Paths.from_env()
k100k_uni2 = paths.embedding("kather100k", "uni2")
k100k_uni2_train, k100k_uni2_test = k100k_uni2.load_splits()

From the `README` here is how the projector callibration works:

For each of projectors generate a set of hyperparameter configurations by the grid.
At each of the *callibration dimensions* and for each of the projector, for each hyperparameter configuration, for each fold of the data:
1. fit on the train of the fold
2. transform the train and valid of the fold
3. return the k-nn accuracy for the valid. Note - The k-nn *score* is computed of *values of k* and averaged.
The mean of the folds to find the k-nn score cross validation value for that configuration at that dimension.
Select configuration that performs best across all dimensions and across all datasets (take the mean score over all dimensions for each configurations).
Repeat this *n-times* for stochastic projectors and take the best of the n runs. This is using paired seeds, so 
have a set of seed defined upfront and reuse them multiple times.

So we end up with a configuration task that has `(dataset, dim, config, fold, seed)` as it's input and a score as the output.

In [2]:
# UMAP takes a long time to run on the full dataset, so we will use a stratified sample

ten_percent = k100k_uni2_train.num_samples() * 0.1
per_class = int(ten_percent / k100k_uni2_train.num_classes())
sampled_k100k_uni2_train = k100k_uni2_train.stratified_sample(per_class)
sampled_k100k_uni2_train.describe()

{'features': '(9999, 1536), float32',
 'labels': '(9999,), int64',
 'metrics': {}}

In [3]:
from statistics import mean

from hproj.data.feature_space import FeatureSpace
from hproj.data.folds import Fold, generate_stratified_folds
from hproj.measure.measurement import Measurement
from hproj.measure.knn_score import KNNMeanScore
from hproj.projectors.projector import Projector
from hproj.projectors.umap import UMAPProjector


def score_projector_config(
    embeddings: FeatureSpace,
    projector: Projector,
    measurement: Measurement,
    folds: list[Fold],
) -> float:
    # this function scores a projector (that has been configured with some hyperparameters)
    # by applying it to the given embeddings and measuring the quality of the resulting projection with the given measurement.
    # The score is averaged across the given folds.
    fold_scores = []
    for train, valid in embeddings.get_folds(folds):
        projector.fit(train)
        train_proj = projector.transform(train)
        valid_proj = projector.transform(valid)
        score = measurement(train_proj, valid_proj)
        fold_scores.append(score)
    return float(mean(fold_scores))


seed = 42
folds = generate_stratified_folds(sampled_k100k_uni2_train, n_splits=5, seed=seed)

# n_components and seed are not hyperparameters of the projector
n_components = 2

# here we configure the projector with some hyperparameters and score it with the KNNMeanScore measurement.
hyperparams = {
    "n_neighbors": 15,
    "min_dist": 0.1,
}

# we use the same folds for all projector configurations, so we generate them once here.
projector = UMAPProjector(n_components, seed=seed, **hyperparams)
measurement = KNNMeanScore([5, 15, 30])
score = score_projector_config(sampled_k100k_uni2_train, projector, measurement, folds)
print(f"Mean K-NN Score for UMAP with hyperparameters {hyperparams}: {score}")

Mean K-NN Score for UMAP with hyperparameters {'n_neighbors': 15, 'min_dist': 0.1}: 0.9883322994830749


Now we function that can score a hyperparameter configuration for a projector, how do we do the search and how do we do it in Parrallel?
- First this to do is to work out all the permutations of the parameters.
- Second thing to do is to distribute the work over the different tasks (cores / gpus)

In [4]:
from hproj.util.hyperparams import make_param_grid
from time import perf_counter

# note we are reusing sampled_k100k_uni2_train and folds from above

# generate a grid of hyperparameters to search over
grid_spec = {
    "n_neighbors": [15, 30, 50],
    "min_dist": [0.0, 0.1, 0.5],
    "metric": ["cosine"],
}
param_grid = make_param_grid(grid_spec)
print(f"Generated {len(param_grid)} hyperparameter combinations to evaluate.")

# iterate over the grid and score each configuration
measurement = KNNMeanScore([5, 15, 30])
results = []
start_time = perf_counter()

for hyperparams in param_grid:
    projector = UMAPProjector(n_components, seed=seed, **hyperparams)
    score = score_projector_config(
        sampled_k100k_uni2_train, projector, measurement, folds
    )
    results.append((hyperparams, score))
    print(f"Mean K-NN Score for UMAP with hyperparameters {hyperparams}: {score}")

elapsed_time = perf_counter() - start_time
print(f"\nTotal time: {elapsed_time:.2f} seconds")

Generated 9 hyperparameter combinations to evaluate.
Mean K-NN Score for UMAP with hyperparameters {'n_neighbors': 15, 'min_dist': 0.0, 'metric': 'cosine'}: 0.9903991829247958
Mean K-NN Score for UMAP with hyperparameters {'n_neighbors': 15, 'min_dist': 0.1, 'metric': 'cosine'}: 0.9902660163415041
Mean K-NN Score for UMAP with hyperparameters {'n_neighbors': 15, 'min_dist': 0.5, 'metric': 'cosine'}: 0.9897325829581458
Mean K-NN Score for UMAP with hyperparameters {'n_neighbors': 30, 'min_dist': 0.0, 'metric': 'cosine'}: 0.9891989494747374
Mean K-NN Score for UMAP with hyperparameters {'n_neighbors': 30, 'min_dist': 0.1, 'metric': 'cosine'}: 0.9892323328330832
Mean K-NN Score for UMAP with hyperparameters {'n_neighbors': 30, 'min_dist': 0.5, 'metric': 'cosine'}: 0.9892657328664333
Mean K-NN Score for UMAP with hyperparameters {'n_neighbors': 50, 'min_dist': 0.0, 'metric': 'cosine'}: 0.988232316158079
Mean K-NN Score for UMAP with hyperparameters {'n_neighbors': 50, 'min_dist': 0.1, 'met

Now let's try and distibute it using and see if that speeds things up.

In [ ]:
from dask.distributed import Client
from dask_cuda import LocalCUDACluster

from hproj.util.hyperparams import make_param_grid


def score_projector_config_task(
    embeddings: FeatureSpace,
    projector_cls: type[Projector],
    n_components: int,
    seed: int,
    hyperparams: dict,
    measurement: Measurement,
    folds: list[Fold],
) -> dict:
    projector = projector_cls(
        n_components=n_components,
        seed=seed,
        **hyperparams,
    )

    score = score_projector_config(
        embeddings=embeddings,
        projector=projector,
        measurement=measurement,
        folds=folds,
    )

    return {
        "hyperparams": hyperparams,
        "score": score,
    }


def evaluate_param_grid(
    client,
    feature_space_train: FeatureSpace,
    n_components: int,
    seed: int,
    projector_cls: type[Projector],
    param_grid: list[dict],
    folds: list[Fold],
    measurement: Measurement,
):
    fs_future = client.scatter(feature_space_train, broadcast=False)

    futures = [
        client.submit(
            score_projector_config_task,
            fs_future,
            projector_cls,
            n_components,
            seed,
            hyperparams,
            measurement,
            folds,
            pure=False,
        )
        for hyperparams in param_grid
    ]

    return client.gather(futures)


def make_dask_client():
    cluster = LocalCUDACluster(
        threads_per_worker=1,
    )
    client = Client(cluster)
    return client, cluster


n_components = 2
seed = 42

# generate a grid of hyperparameters to search over
grid_spec = {
    "n_neighbors": [2, 5, 10, 20, 50, 100, 200],
    "min_dist": [0.0, 0.1, 0.25, 0.5, 0.8, 0.99],
    "metric": ["cosine"],
}
param_grid = make_param_grid(grid_spec)
print(f"Generated {len(param_grid)} hyperparameter combinations to evaluate.")

# iterate over the grid and score each configuration
measurement = KNNMeanScore([5, 15, 30])

start_time = perf_counter()

client, cluster = make_dask_client()
results = evaluate_param_grid(
    client,
    sampled_k100k_uni2_train,
    n_components,
    seed,
    UMAPProjector,
    param_grid,
    folds,
    measurement,
)

for result in results:
    hyperparams = result["hyperparams"]
    score = result["score"]
    print(f"Mean K-NN Score for UMAP with hyperparameters {hyperparams}: {score}")

client.close()
cluster.close()

elapsed_time = perf_counter() - start_time
print(f"\nTotal time: {elapsed_time:.2f} seconds")

Generated 42 hyperparameter combinations to evaluate.
Mean K-NN Score for UMAP with hyperparameters {'n_neighbors': 2, 'min_dist': 0.0, 'metric': 'cosine'}: 0.4550110221777555
Mean K-NN Score for UMAP with hyperparameters {'n_neighbors': 2, 'min_dist': 0.1, 'metric': 'cosine'}: 0.46201277305319327
Mean K-NN Score for UMAP with hyperparameters {'n_neighbors': 2, 'min_dist': 0.25, 'metric': 'cosine'}: 0.4616465899616475
Mean K-NN Score for UMAP with hyperparameters {'n_neighbors': 2, 'min_dist': 0.5, 'metric': 'cosine'}: 0.4577791895947974
Mean K-NN Score for UMAP with hyperparameters {'n_neighbors': 2, 'min_dist': 0.8, 'metric': 'cosine'}: 0.42384212106053026
Mean K-NN Score for UMAP with hyperparameters {'n_neighbors': 2, 'min_dist': 0.99, 'metric': 'cosine'}: 0.3999737535434384
Mean K-NN Score for UMAP with hyperparameters {'n_neighbors': 5, 'min_dist': 0.0, 'metric': 'cosine'}: 0.9863652826413206
Mean K-NN Score for UMAP with hyperparameters {'n_neighbors': 5, 'min_dist': 0.1, 'metri

So the cost of distributing it is to much.